# Module 7.2 — Make DeskMate an Agent

**Goal:** Extend DeskMate with tool-calling so it can look up an order, file a bug report, or escalate a ticket as real actions — and add a loop guard to prevent infinite agent loops. Measure when agent mode improves accuracy vs plain RAG.

## 1. Setup

In [ ]:
import os, re, json, time, logging
import matplotlib.pyplot as plt
import numpy as np

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
logger = logging.getLogger('deskmate.agent')

SMOKE_TEST  = True
MAX_TURNS   = 5
MAX_CONTEXT = 4096

os.makedirs('reports', exist_ok=True)
print('Config OK')

## 2. Tool Definitions

In [ ]:
TOOLS = [
    {
        'type': 'function',
        'function': {
            'name': 'lookup_order',
            'description': 'Look up a customer order by order ID. Returns status, carrier, and ETA.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'order_id': {'type': 'string', 'description': 'Order ID, e.g. ORD-7293'},
                },
                'required': ['order_id'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'file_bug',
            'description': 'File a bug report. Returns the new issue ID and tracker URL.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'title':       {'type': 'string'},
                    'description': {'type': 'string'},
                    'severity':    {'type': 'string', 'enum': ['low','medium','high','critical']},
                },
                'required': ['title', 'description', 'severity'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'escalate_to_human',
            'description': 'Escalate to a human agent when the issue cannot be resolved automatically.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'reason':   {'type': 'string'},
                    'priority': {'type': 'string', 'enum': ['normal','high','urgent']},
                },
                'required': ['reason', 'priority'],
            },
        },
    },
]

tool_names = [t['function']['name'] for t in TOOLS]
print('Tools:', tool_names)

## 3. Tool Implementations (Stubs)

In [ ]:
ORDER_DB = {
    'ORD-7293': {'status': 'shipped', 'carrier': 'FedEx',  'tracking': 'FX-829341', 'eta': '2026-06-28'},
    'ORD-1001': {'status': 'delivered','carrier': 'UPS',    'tracking': 'UP-112233', 'eta': '2026-06-24'},
    'ORD-5555': {'status': 'processing','carrier': None,    'tracking': None,         'eta': '2026-07-02'},
}

def lookup_order(order_id: str) -> dict:
    rec = ORDER_DB.get(order_id.upper())
    if not rec:
        return {'error': f'Order {order_id} not found.'}
    return {'order_id': order_id, **rec}

def file_bug(title: str, description: str, severity: str) -> dict:
    issue_id = f"BUG-{abs(hash(title)) % 10000:04d}"
    return {
        'issue_id': issue_id,
        'status':   'created',
        'severity': severity,
        'url':      f'https://issues.deskmate.com/{issue_id}',
    }

def escalate_to_human(reason: str, priority: str) -> dict:
    return {'escalated': True, 'priority': priority,
            'ticket_id': 'ESC-9001', 'message': 'Human agent will contact you within 4 hours.'}

TOOL_FNS = {
    'lookup_order':     lookup_order,
    'file_bug':         file_bug,
    'escalate_to_human':escalate_to_human,
}

# Quick test
print('lookup_order:', lookup_order('ORD-7293'))
print('file_bug:',     file_bug('CSV export crash', 'Export fails with ERR-500', 'high'))
print('escalate:',     escalate_to_human('Cannot verify account', 'urgent'))

## 4. Tool Dispatcher

In [ ]:
def dispatch_tool(name: str, args: dict) -> dict:
    fn = TOOL_FNS.get(name)
    if fn is None:
        return {'error': f'Unknown tool: {name}'}
    try:
        return fn(**args)
    except Exception as e:
        logger.error('Tool %s failed: %s', name, e)
        return {'error': str(e)}

# Test dispatcher
result = dispatch_tool('lookup_order', {'order_id': 'ORD-7293'})
print('Dispatch result:', result)

## 5. LLM Stub (Simulates Function-Calling Response)

In [ ]:
import dataclasses

@dataclasses.dataclass
class ToolCall:
    id: str
    name: str
    arguments: str  # JSON string

@dataclasses.dataclass
class FakeMessage:
    content: str | None
    tool_calls: list | None = None

def llm_agent_stub(messages: list, tools: list) -> FakeMessage:
    last_user = next((m['content'] for m in reversed(messages)
                      if m.get('role') == 'user'), '')
    last_tool = next((m.get('content','') for m in reversed(messages)
                      if m.get('role') == 'tool'), None)

    # If we just got a tool result, write final reply
    if last_tool is not None:
        try:
            data = json.loads(last_tool)
        except Exception:
            data = {}
        if data.get('status') == 'shipped':
            return FakeMessage(
                content='Your order ' + data.get('order_id','') + ' has been shipped via '
                + data.get('carrier','') + '. Tracking: ' + (data.get('tracking') or '') + '. ETA: ' + data.get('eta','') + '. [Source 1]')
        if data.get('issue_id'):
            return FakeMessage(
                content='I have filed a bug report ' + data['issue_id'] + ' for this issue. '
                + 'Our team will investigate. You can track it at ' + data.get('url','') + '.')
        if data.get('escalated'):
            return FakeMessage(
                content='Your ticket has been escalated (priority: ' + data.get('priority','') + '). '
                + data.get('message',''))
        return FakeMessage(content='I have processed your request. '
                           + json.dumps(data))

    # Decide which tool to call based on ticket content
    ticket = last_user.lower()
    order_match = re.search(r'ord-\d+', ticket, re.IGNORECASE)
    if order_match:
        oid = order_match.group(0).upper()
        return FakeMessage(
            content=None,
            tool_calls=[ToolCall(
                id='call_1', name='lookup_order',
                arguments=json.dumps({'order_id': oid}))])
    if 'csv' in ticket or 'export' in ticket or 'crash' in ticket:
        return FakeMessage(
            content=None,
            tool_calls=[ToolCall(
                id='call_2', name='file_bug',
                arguments=json.dumps({'title': 'CSV export crash',
                                      'description': last_user[:200],
                                      'severity': 'high'}))])
    # No tool needed
    return FakeMessage(content='Based on our documentation, '
                       + 'please contact support@deskmate.com for further assistance. [Source 1]')

def parse_tool_call(response):
    if not response.tool_calls:
        return None
    tc = response.tool_calls[0]
    return {'name': tc.name, 'args': json.loads(tc.arguments), 'id': tc.id}

print('LLM stub ready.')

## 6. Initial Message Builder

In [ ]:
SYSTEM_AGENT = (
    'You are DeskMate, a support agent. You have access to tools. '
    'Use tools to fetch real data when needed. '
    'Always cite [Source N] when using context passages. '
    'Call escalate_to_human if you cannot resolve the issue.'
)

def build_initial_messages(ticket: str, chunks: list) -> list:
    ctx = '\n\n'.join(
        f"[Source {i+1}] {c['source']}\n{c['text']}" for i, c in enumerate(chunks))
    user_content = f'Context:\n{ctx}\n\nTicket: {ticket}' if ctx else f'Ticket: {ticket}'
    return [
        {'role': 'system', 'content': SYSTEM_AGENT},
        {'role': 'user',   'content': user_content},
    ]

# Dummy chunks for testing
TEST_CHUNKS = [
    {'source': 'shipping_policy.md', 'text': 'Standard shipping takes 3-5 business days.'},
    {'source': 'billing_refunds.md', 'text': 'Refunds are processed within 3 business days.'},
]

msgs = build_initial_messages('Where is my order ORD-7293?', TEST_CHUNKS)
print('Messages:', len(msgs), 'role(s):', [m['role'] for m in msgs])

## 7. Single-Step Tool Call

In [ ]:
ticket_order = 'Where is my order ORD-7293? It was supposed to arrive yesterday.'
msgs = build_initial_messages(ticket_order, TEST_CHUNKS)

print('=== Step 1: First LLM call ===')
resp1 = llm_agent_stub(msgs, TOOLS)
tc    = parse_tool_call(resp1)
print('Tool call:', tc)

print('\n=== Step 2: Execute tool ===')
tool_result = dispatch_tool(tc['name'], tc['args'])
print('Tool result:', tool_result)

print('\n=== Step 3: Second LLM call with tool result ===')
msgs.append({'role': 'assistant', 'content': None,
             'tool_calls': [{'id': tc['id'], 'type': 'function',
                             'function': {'name': tc['name'],
                                          'arguments': json.dumps(tc['args'])}}]})
msgs.append({'role': 'tool', 'tool_call_id': tc['id'],
             'content': json.dumps(tool_result)})
resp2 = llm_agent_stub(msgs, TOOLS)
print('Final reply:', resp2.content)

## 8. Full Agent Loop

In [ ]:
def agent_loop(ticket: str, chunks: list) -> dict:
    messages = build_initial_messages(ticket, chunks)
    turns = 0
    tool_log = []

    while turns < MAX_TURNS:
        # Context-length guard
        approx_tokens = sum(len(str(m.get('content',''))).split if False else [len(str(m.get('content','')))] for m in messages)
        approx_tokens = sum(len(str(m.get('content',''))) for m in messages) // 4
        if approx_tokens > MAX_CONTEXT:
            return {'action': 'error', 'reason': 'context_limit',
                    'reply': 'Request too long. Please contact support.',
                    'turns': turns, 'tool_log': tool_log}

        response = llm_agent_stub(messages, TOOLS)
        tc = parse_tool_call(response)

        if tc is None:  # done
            return {'action': 'reply', 'reply': response.content,
                    'turns': turns, 'tool_log': tool_log}

        tool_log.append({'tool': tc['name'], 'args': tc['args']})
        result = dispatch_tool(tc['name'], tc['args'])
        tool_log[-1]['result'] = result

        messages.append({'role': 'assistant', 'content': None,
                         'tool_calls': [{'id': tc['id'], 'type': 'function',
                                          'function': {'name': tc['name'],
                                                       'arguments': json.dumps(tc['args'])}}]})
        messages.append({'role': 'tool', 'tool_call_id': tc['id'],
                         'content': json.dumps(result)})
        turns += 1

    # Loop guard fired
    logger.warning('agent_loop: MAX_TURNS=%d exceeded for ticket: %s', MAX_TURNS, ticket[:60])
    return {
        'action': 'escalate',
        'reason': f'loop_limit_exceeded (>{MAX_TURNS} turns)',
        'reply':  'I was unable to resolve this automatically. A human agent will follow up.',
        'turns':  turns,
        'tool_log': tool_log,
    }

# Test: order lookup
result = agent_loop('Where is my order ORD-7293?', TEST_CHUNKS)
print('Action:', result['action'])
print('Reply:', result['reply'])
print('Turns:', result['turns'])
print('Tool log:', result['tool_log'])

## 9. Loop Guard Test

In [ ]:
# Force loop guard by making the stub always return a tool call
def llm_always_calls_tool(messages, tools):
    return FakeMessage(
        content=None,
        tool_calls=[ToolCall(id='c1', name='lookup_order',
                             arguments=json.dumps({'order_id': 'ORD-9999'}))])

def agent_loop_with_stub(ticket, chunks, stub_fn):
    messages = build_initial_messages(ticket, chunks)
    turns, tool_log = 0, []
    while turns < MAX_TURNS:
        response = stub_fn(messages, TOOLS)
        tc = parse_tool_call(response)
        if tc is None:
            return {'action': 'reply', 'reply': response.content, 'turns': turns}
        tool_log.append(tc['name'])
        result = dispatch_tool(tc['name'], tc['args'])
        messages.append({'role': 'assistant', 'content': None,
                         'tool_calls': [{'id': tc['id'], 'type': 'function',
                                          'function': {'name': tc['name'],
                                                       'arguments': json.dumps(tc['args'])}}]})
        messages.append({'role': 'tool', 'tool_call_id': tc['id'],
                         'content': json.dumps(result)})
        turns += 1
    return {'action': 'escalate', 'reason': 'loop_limit', 'turns': turns, 'tool_log': tool_log}

result = agent_loop_with_stub('force loop', [], llm_always_calls_tool)
print('Action:', result['action'])
print('Turns:',  result['turns'])
print('Loop guard fired correctly:', result['action'] == 'escalate')
assert result['action'] == 'escalate', 'Loop guard did not fire!'
assert result['turns'] == MAX_TURNS,   'Wrong turn count!'
print('PASS: Loop guard test')

## 10. Context-Limit Guard Test

In [ ]:
def agent_loop_ctx_test(ticket, huge_chunks):
    messages = build_initial_messages(ticket, huge_chunks)
    approx_tokens = sum(len(str(m.get('content',''))) for m in messages) // 4
    if approx_tokens > MAX_CONTEXT:
        return {'action': 'error', 'reason': 'context_limit', 'approx_tokens': approx_tokens}
    return {'action': 'continue'}

# Create huge chunks that overflow the context limit
huge_chunks = [{'source': f'doc_{i}.md', 'text': 'X ' * 2000} for i in range(5)]
result = agent_loop_ctx_test('test ticket', huge_chunks)
print('Result:', result)
print('Context guard fired correctly:', result['action'] == 'error')
assert result['action'] == 'error'
print('PASS: Context-limit guard test')

## 11. Batch Evaluation — Agent vs RAG

In [ ]:
from rouge_score import rouge_scorer as rs_module
_scorer = rs_module.RougeScorer(['rougeL'], use_stemmer=True)

GOLD_AGENT = [
    {'ticket': 'Where is my order ORD-7293? It has not arrived.',
     'ref': 'Your order ORD-7293 has been shipped via FedEx. Tracking: FX-829341. ETA: 2026-06-28.',
     'needs_tool': True},
    {'ticket': 'My order ORD-1001 was supposed to arrive yesterday.',
     'ref': 'Your order ORD-1001 has been delivered via UPS. Tracking: UP-112233.',
     'needs_tool': True},
    {'ticket': 'The CSV export crashes with ERR-500.',
     'ref': 'I have filed bug report BUG-',
     'needs_tool': True},
    {'ticket': 'I was charged twice for my subscription.',
     'ref': 'Contact support with your invoice numbers for a billing dispute.',
     'needs_tool': False},
    {'ticket': 'How do I reset my password?',
     'ref': 'Go to the login page and click Forgot password.',
     'needs_tool': False},
]

# RAG baseline (stateless, no tools)
def rag_reply_stub(ticket):
    if 'order' in ticket.lower() or 'ORD' in ticket:
        return 'Please check your email for shipping updates. [Source 1]'
    if 'csv' in ticket.lower() or 'export' in ticket.lower():
        return 'This is a known issue fixed in version 4.3.0. [Source 1]'
    return 'Please contact support@deskmate.com. [Source 1]'

results = []
for ex in GOLD_AGENT:
    agent_r  = agent_loop(ex['ticket'], TEST_CHUNKS)
    rag_rep  = rag_reply_stub(ex['ticket'])

    a_rouge = _scorer.score(ex['ref'], agent_r.get('reply') or '')['rougeL'].fmeasure
    r_rouge = _scorer.score(ex['ref'], rag_rep)['rougeL'].fmeasure
    results.append({
        'ticket_short': ex['ticket'][:45],
        'needs_tool':   ex['needs_tool'],
        'agent_rouge':  round(a_rouge, 3),
        'rag_rouge':    round(r_rouge, 3),
        'agent_turns':  agent_r.get('turns', 0),
    })

print(f'{"Ticket":<46} Needs tool  Agent ROUGE  RAG ROUGE  Turns')
print('-' * 90)
for r in results:
    print(f"{r['ticket_short']:<46} {str(r['needs_tool']):<11} "
          f"{r['agent_rouge']:<13} {r['rag_rouge']:<10} {r['agent_turns']}")

## 12. Comparison Chart

In [ ]:
tool_tickets    = [r for r in results if r['needs_tool']]
nontool_tickets = [r for r in results if not r['needs_tool']]

def avg(lst, key): return round(sum(r[key] for r in lst) / len(lst), 3) if lst else 0

categories    = ['Tool-needed tickets', 'Tool-not-needed tickets']
agent_rouges  = [avg(tool_tickets, 'agent_rouge'), avg(nontool_tickets, 'agent_rouge')]
rag_rouges    = [avg(tool_tickets, 'rag_rouge'),   avg(nontool_tickets, 'rag_rouge')]

x   = np.arange(len(categories))
w   = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, agent_rouges, w, label='Agent (tool-calling)', color='steelblue')
ax.bar(x + w/2, rag_rouges,   w, label='Stateless RAG',        color='coral')
ax.set_ylabel('ROUGE-L')
ax.set_title('Agent vs Stateless RAG — by ticket type')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()
ax.set_ylim(0, 1.0)
for i, v in enumerate(agent_rouges):
    ax.text(i - w/2, v + 0.02, str(v), ha='center', fontsize=9)
for i, v in enumerate(rag_rouges):
    ax.text(i + w/2, v + 0.02, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('agent_vs_rag.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: agent_vs_rag.png')

## 13. Checkpoint

In [ ]:
checkpoint = (
    'What guardrail prevents an agent tool-call loop from running forever?\n\n'
    'A turn counter (MAX_TURNS) checked at the top of every loop iteration.\n'
    'When tool calls reach MAX_TURNS, the loop exits and escalates to a human\n'
    'rather than returning a fabricated reply.\n\n'
    'Three complementary guards:\n'
    '  1. Turn counter (MAX_TURNS=5): primary guard; fires on excessive tool use.\n'
    '  2. Context-length ceiling: stops loop before prompt overflows the model window.\n'
    '  3. HTTP timeout at the API layer: terminates requests that exceed wall-clock time.\n\n'
    'MAX_TURNS is calibrated to the maximum legitimate tool depth for the use case.\n'
    'DeskMate needs at most 2 tool calls per request, so MAX_TURNS=5 gives headroom\n'
    'without allowing runaway loops.'
)
print(checkpoint)

## 14. Save Report

In [ ]:
tool_delta  = round(avg(tool_tickets,'agent_rouge') - avg(tool_tickets,'rag_rouge'), 3)
plain_delta = round(avg(nontool_tickets,'agent_rouge') - avg(nontool_tickets,'rag_rouge'), 3)

report = [
    '# Agent Report\n',
    f'Smoke test: {SMOKE_TEST}',
    f'MAX_TURNS: {MAX_TURNS}\n',
    '## ROUGE-L Summary',
    '',
    f'Tool-needed tickets  — Agent: {avg(tool_tickets,"agent_rouge")}  '
    + f'RAG: {avg(tool_tickets,"rag_rouge")}  Delta: +{tool_delta}',
    f'Tool-not-needed      — Agent: {avg(nontool_tickets,"agent_rouge")}  '
    + f'RAG: {avg(nontool_tickets,"rag_rouge")}  Delta: {plain_delta}',
    '',
    '## Loop Guard Tests',
    '',
    'Turn counter (MAX_TURNS=5): PASS',
    'Context-limit guard: PASS',
    '',
    '## Checkpoint',
    '',
    'Loop guard = turn counter MAX_TURNS. Fires after 5 tool calls; escalates to human.',
    'Complementary: context-length ceiling + HTTP timeout.',
    'MAX_TURNS=5 chosen because DeskMate needs at most 2 calls per valid request.',
]

with open('reports/agent_report.md', 'w') as f:
    f.write('\n'.join(report))

print('Saved: reports/agent_report.md')
print('\n=== Module 7.2 Complete ===')